# 02 — Symbolic verifier analysis: MC option-proof + statement-form

Analysis-only. It consumes rebuild/v35 preds when available, and produces candidate proof reports. No correction is applied here.

In [ ]:

import json, re, os, math, csv, statistics
from pathlib import Path
from collections import Counter, defaultdict

CANDIDATE_DIRS = [Path('/kaggle/working'), Path('/kaggle/input/datasets/yixuanisthebest/v2333333'), Path('/kaggle/input/datasets/nguyenminhtric/test-pipeline'), Path('/mnt/data')]

def find_file(names, required=True):
    if isinstance(names, str): names=[names]
    for name in names:
        p=Path(name)
        if p.exists(): return p
    for d in CANDIDATE_DIRS:
        if not d.exists(): continue
        for name in names:
            p=d/name
            if p.exists(): return p
    if required:
        raise FileNotFoundError(f'Could not find any of {names} in {CANDIDATE_DIRS}')
    return None

def load_json(names, required=True):
    p=find_file(names, required=required)
    if p is None: return None, None
    with open(p,'r',encoding='utf-8') as f: return json.load(f), p

def save_json(obj, path):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path,'w',encoding='utf-8') as f: json.dump(obj,f,ensure_ascii=False,indent=2)
    # reload verify
    with open(path,'r',encoding='utf-8') as f: json.load(f)
    print('saved', path)

def norm_text(s): return re.sub(r'\s+',' ',str(s).strip())
def norm_key(s): return re.sub(r'[^a-z0-9]+',' ',str(s).lower()).strip()

def camel_to_words(s):
    s=re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    return norm_key(s)

def label_norm(x):
    if x is None: return None
    s=str(x).strip()
    if s.upper() in ['A','B','C','D']: return s.upper()
    t=s.title()
    return t if t in ['Yes','No','Unknown','Unparseable'] else s

def is_mc_question(q):
    return bool(re.search(r'\n\s*A\.', str(q))) or all(x in str(q) for x in ['A.','B.'])

def flatten_dataset(rows, dataset_name='dataset'):
    flat=[]
    for ri,row in enumerate(rows):
        qs=row.get('questions') if isinstance(row,dict) else None
        ans=row.get('answers') if isinstance(row,dict) else None
        exps=row.get('explanation') if isinstance(row,dict) else None
        if isinstance(qs, list):
            for qi,q in enumerate(qs):
                flat.append({
                    'flat_id': f'{dataset_name}_{ri}_{qi}',
                    'row_i': ri, 'q_i': qi,
                    'question': q,
                    'gold': label_norm(ans[qi]) if isinstance(ans,list) and qi < len(ans) else None,
                    'explanation': exps[qi] if isinstance(exps,list) and qi < len(exps) else None,
                    'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                    'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                    'idx': row.get('idx'),
                    'is_mc': is_mc_question(q),
                    'is_ynu': not is_mc_question(q),
                })
        elif isinstance(row, dict):
            q=row.get('question') or row.get('query') or ''
            flat.append({
                'flat_id': f'{dataset_name}_{ri}_0', 'row_i': ri, 'q_i': 0,
                'question': q, 'gold': label_norm(row.get('answer') or row.get('gold')),
                'explanation': row.get('explanation'),
                'premises_FOL': row.get('premises-FOL') or row.get('premises_FOL') or [],
                'premises_NL': row.get('premises-NL') or row.get('premises_NL') or [],
                'idx': row.get('idx'), 'is_mc': is_mc_question(q), 'is_ynu': not is_mc_question(q)
            })
    return flat

ATOM_RE = re.compile(r'¬?([A-Z][A-Za-z0-9_]*)\(x\)')

def parse_fol_items(fols):
    facts_pos=set(); facts_neg=set(); exists_pos=set(); exists_neg=set(); impl=[]; raw=[]
    for s in fols or []:
        t=str(s).replace('→','->').replace('∀','forall').replace('∃','exists').replace('¬','~')
        raw.append(str(s))
        # exists conjunctions
        if 'exists' in t:
            for m in re.finditer(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t):
                (exists_neg if m.group(1) else exists_pos).add(m.group(2))
        elif 'forall' in t and '->' not in t:
            m=re.search(r'forall\s*x\s*\(?\s*(~?)([A-Z][A-Za-z0-9_]*)\(x\)', t)
            if m:
                (facts_neg if m.group(1) else facts_pos).add(m.group(2))
        if 'forall' in t and '->' in t:
            parts=t.split('->',1)
            left=parts[0]; right=parts[1]
            ml=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', left)
            mr=re.findall(r'(~?)([A-Z][A-Za-z0-9_]*)\(x\)', right)
            if ml and mr:
                a_neg,a=ml[-1]; b_neg,b=mr[0]
                impl.append((a, bool(a_neg), b, bool(b_neg), str(s)))
    return {'facts_pos':facts_pos,'facts_neg':facts_neg,'exists_pos':exists_pos,'exists_neg':exists_neg,'impl':impl,'raw':raw}

def closure(parsed):
    pos=set(parsed['facts_pos']); neg=set(parsed['facts_neg']); paths={('pos',p):f'GIVEN ∀x {p}' for p in pos}; paths.update({('neg',p):f'GIVEN ∀x ¬{p}' for p in neg})
    changed=True
    while changed:
        changed=False
        for a,an,b,bn,raw in parsed['impl']:
            # forward
            if not an and a in pos:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            if an and a in neg:
                target=neg if bn else pos; key=('neg' if bn else 'pos', b)
                if b not in target:
                    target.add(b); paths[key]=raw; changed=True
            # contrapositive for single-antecedent implication: A -> B gives ~B -> ~A
            if not an and not bn and b in neg and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
            if not an and bn and b in pos and a not in neg:
                neg.add(a); paths[('neg',a)]='CONTRAPOSITIVE_OF: '+raw; changed=True
    return pos,neg,paths

def target_from_question(q):
    # approximate by matching CamelCase words against question tokens; fallback none
    return None

def predicate_candidates(fols):
    preds=set()
    for s in fols or []:
        preds.update(re.findall(r'([A-Z][A-Za-z0-9_]*)\(x\)', str(s)))
    return sorted(preds)

def best_predicate_match(text, predicates):
    txt=norm_key(text)
    best=(0,None)
    for p in predicates:
        words=camel_to_words(p).split()
        if not words: continue
        hit=sum(1 for w in words if w in txt)
        score=hit/len(words)
        # exact full phrase bonus
        if camel_to_words(p) in txt: score=max(score,1.0)
        if score>best[0]: best=(score,p)
    return best[1], best[0]

def qtype(q):
    s=str(q).lower()
    if is_mc_question(q): return 'mc'
    if 'at least one' in s or 'there is at least one' in s or re.search(r'\bsome\b', s): return 'existential'
    if re.search(r'\b(all|every)\b', s): return 'universal'
    if 'statement:' in s and 'if ' in s and ' then ' in s: return 'statement_conditional'
    if 'no ' in s: return 'universal_negative'
    return 'other_ynu'

def answer_from_explanation(exp):
    if not exp: return None
    m=re.findall(r'(?:answer is|final answer\s*[:\-]?)\s*(Yes|No|Unknown|[ABCD])\b', str(exp), flags=re.I)
    return label_norm(m[-1]) if m else None


In [ ]:

OUT_DIR=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
TEST_FILES={'benchmark_v2_1000':'benchmark_v2_1000.json','generated_v4style_300':'generated_v4style_300.json'}
PRED_FILES={
 'benchmark_v2_1000': ['rebuild_benchmark_v2_1000_preds.json','v35_benchmark_v2_1000_preds.json'],
 'generated_v4style_300': ['rebuild_generated_v4style_300_preds.json','v35_fulltest_preds.json','full_model_eval_v3_2_preds.json'],
}


In [ ]:

def option_blocks(q):
    q=str(q)
    blocks={}
    for lab in 'ABCD':
        m=re.search(rf'{lab}\.\s*(.*?)(?=\n[A-D]\.\s*|$)', q, flags=re.S)
        if m: blocks[lab]=norm_text(m.group(1))
    return blocks

def proof_for_text(text, fols):
    parsed=parse_fol_items(fols); pos,neg,paths=closure(parsed); preds=predicate_candidates(fols)
    target,score=best_predicate_match(text,preds)
    low=str(text).lower()
    polarity='pos'
    quant='unknown'
    if re.search(r'\b(no|not|cannot|insufficient|never|without)\b', low): polarity='neg'
    if re.search(r'\b(all|every)\b', low): quant='universal'
    elif 'at least one' in low or re.search(r'\bsome\b', low): quant='existential'
    elif low.startswith('if '): quant='conditional'
    proven=False; path=None
    if target and score>=0.66:
        if polarity=='pos' and target in pos:
            proven=True; path=paths.get(('pos',target))
        if polarity=='neg' and target in neg:
            proven=True; path=paths.get(('neg',target))
        # existential positive can be proven by exists or universal positive
        if polarity=='pos' and quant=='existential' and (target in parsed['exists_pos'] or target in pos):
            proven=True; path=path or ('EXISTS_OR_UNIVERSAL: '+target)
    return {'target':target,'target_score':score,'polarity':polarity,'quant':quant,'proven':proven,'path':path}

def analyze_mc(flat, pred=None):
    opts=option_blocks(flat['question'])
    proofs={lab:proof_for_text(txt, flat.get('premises_FOL') or []) for lab,txt in opts.items()}
    proven=[lab for lab,p in proofs.items() if p['proven']]
    candidate=None
    if len(proven)==1:
        candidate=proven[0]
    return {'flat_id':flat['flat_id'],'gold':flat.get('gold'),'pred':pred,'question':flat['question'], 'proven_options':proven, 'candidate':candidate, 'proofs':proofs, 'posthoc': (None if candidate is None or flat.get('gold') is None else ('GOOD' if candidate==flat.get('gold') else 'BAD'))}

def analyze_statement(flat, pred=None):
    q=str(flat['question'])
    if 'statement:' not in q.lower(): return None
    stmt=q.split('Statement:',1)[-1].strip() if 'Statement:' in q else q
    pr=proof_for_text(stmt, flat.get('premises_FOL') or [])
    candidate=None
    if pr['proven']:
        candidate='No' if pr['polarity']=='neg' else 'Yes'
    return {'flat_id':flat['flat_id'],'gold':flat.get('gold'),'pred':pred,'question':q,'statement':stmt,'candidate':candidate,'proof':pr,'posthoc': (None if candidate is None or flat.get('gold') is None else ('GOOD' if candidate==flat.get('gold') else 'BAD'))}

def load_preds_map(dataset_name):
    rows,p=load_json(PRED_FILES.get(dataset_name,[]), required=False)
    if rows is None: return {}, None
    mp={}
    for r in rows:
        fid=r.get('flat_id') or r.get('id')
        if fid: mp[fid]=r
    return mp,p

combined={}
for name,fn in TEST_FILES.items():
    rows,path=load_json(fn, required=True)
    flat=flatten_dataset(rows, name)
    preds,pred_path=load_preds_map(name)
    mc_reports=[]; st_reports=[]
    for f in flat:
        pr=preds.get(f['flat_id'],{})
        pred=pr.get('pred_v35') or pr.get('pred_final') or pr.get('pred_v32') or pr.get('pred_parsefix') or pr.get('pred')
        if f['is_mc']:
            mc_reports.append(analyze_mc(f, pred))
        else:
            r=analyze_statement(f,pred)
            if r: st_reports.append(r)
    def summarize(reps):
        cands=[r for r in reps if r.get('candidate')]
        good=sum(r.get('posthoc')=='GOOD' for r in cands); bad=sum(r.get('posthoc')=='BAD' for r in cands)
        return {'n':len(reps),'candidate_n':len(cands),'good':good,'bad':bad,'precision': good/len(cands) if cands else None}
    summary={'dataset':name,'source_path':str(path),'pred_path':str(pred_path) if pred_path else None,'mc_analysis':summarize(mc_reports),'statement_analysis':summarize(st_reports), 'correction_mode':'ANALYSIS_ONLY'}
    save_json(mc_reports, OUT_DIR/f'02_{name}_mc_option_proof_analysis.json')
    save_json(st_reports, OUT_DIR/f'02_{name}_statement_form_analysis.json')
    save_json(summary, OUT_DIR/f'02_{name}_symbolic_analysis_summary.json')
    print('\n',name,json.dumps(summary,indent=2))
    combined[name]=summary
save_json(combined, OUT_DIR/'02_combined_symbolic_analysis_summary.json')
